In [ ]:
import importlib
from .prefetcher import *
from torch.utils.data import DataLoader
from .signaldataset import *
from .preprocess import *

__all__ = ['AMRDataLoader']


class AMRDataLoader(object):
    def __init__(self, dataset, Xmode, batch_size, num_workers, pin_memory, mod_type=[], ismulti=False):

        # 动态导入模块 -> 获取类 -> 实例化 -> 调用
        # importlib.import_module() → 返回模块对象
        # getattr(……,"SignalDataLoader")：返回类对象
        # (mod_type)：实例化类
        # ()：调用实例，假设实现了 __call__
        X_train, Y_train, Z_train, X_valid, Y_valid, Z_valid, X_test, Y_test, Z_test, self.snrs, self.mods = (
            getattr(importlib.import_module("amr.dataloaders.dataloader_" + dataset), "SignalDataLoader")
            (mod_type)
            ()
        )

        # 如果不是多模态输入，使用单一预处理方式
        if not ismulti:
            # 实例化数据预处理类
            datapreprocess = DataPreprocess(Xmode)
        # 多模态输入，为每种模态创建独立的预处理对象
        else:
            datapreprocess = []
            for idx in range(len(Xmode)):
                datapreprocess.append(DataPreprocess(Xmode[idx]))


        # 创建数据集对象
        # 将原始数据和预处理方法封装为PyTorch Dataset
        train_dataset = SignalDataset(X_train, Y_train, Z_train, datapreprocess, ismulti)
        valid_dataset = SignalDataset(X_valid, Y_valid, Z_valid, datapreprocess, ismulti)
        test_dataset = SignalDataset(X_test, Y_test, Z_test, datapreprocess, ismulti)

        self.train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers,
                                       pin_memory=pin_memory, shuffle=True)
        self.valid_loader = DataLoader(valid_dataset, batch_size=batch_size, num_workers=num_workers,
                                       pin_memory=pin_memory, shuffle=False)
        self.test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=num_workers,
                                      pin_memory=pin_memory, shuffle=False)

        if pin_memory:
            self.train_loader = PreFetcher(self.train_loader, ismulti)
            self.valid_loader = PreFetcher(self.valid_loader, ismulti)
            self.test_loader = PreFetcher(self.test_loader, ismulti)

    def __call__(self):
        return self.train_loader, self.valid_loader, self.test_loader, self.snrs, self.mods


if __name__ == '__main__':
    train_loader, valid_loader, test_loader = AMRDataLoader("RML2016.10a_dict.pkl",{"type":"IQ","options":{"IQ_norm":False,"zero_mask":False}},batch_size=100,num_workers=4, pin_memory=False)()
    print(train_loader)
    print(valid_loader)
    print(test_loader)

## AMRDataLoader 与 PreFetcher 的协同工作解析

### 1. PreFetcher 的初始化与调用
在 AMRDataLoader 的 `__init__` 方法最后部分：
```
if pin_memory:
    self.train_loader = PreFetcher(self.train_loader, ismulti)
    self.valid_loader = PreFetcher(self.valid_loader, ismulti)
    self.test_loader = PreFetcher(self.test_loader, ismulti)
```
这段代码展示了 PreFetcher 的使用条件和方法：
- 使用条件：只有当 pin_memory=True 时才会启用预取器
- 包装方式：将原始的 DataLoader 实例包装在 PreFetcher 中
- 参数传递：
    - 第一个参数是原始的数据加载器
    - 第二个参数 ismulti 表示是否是多模态数据

### 2. 工作流程解析
#### 原始数据加载流程（无 PreFetcher）
- 训练循环请求一个 batch
- DataLoader 从磁盘加载数据
- 执行预处理
- 将数据传输到 GPU
- 开始训练
- 重复1-5

#### 使用 PreFetcher 的优化流程
- 训练循环请求当前 batch
- PreFetcher 返回已在 GPU 上的数据
- 同时在后台：
    - PreFetcher 已预取下一个 batch
    - 在另一个 CUDA 流中执行数据传输
- 当前 batch 训练时，下一个 batch 已在准备中

[加载][传输][计算][加载][传输][计算]...

[加载][传输][计算]

      [加载][传输][计算]

           [加载][传输][计算]